In [ ]:
# Cell 0 — allocator config (MUST run before any GPU use) + memory clear
import os, gc
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        fr,to=torch.cuda.mem_get_info(); print(f'VRAM: {fr/1e9:.1f}/{to/1e9:.1f} GB free')
except Exception as e:
    print('torch not yet imported / CUDA not ready:', e)


In [ ]:
# Cell 1 — Mount Google Drive (idempotent: safe to re-run)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
# Cell 2 — Config (single source of truth)
import json
from pathlib import Path
CFG = {
    'drive_tensors':    '/content/drive/MyDrive/DISC5/tensors',
    'drive_tar':        '/content/drive/MyDrive/DISC5/disc5_tensors.tar',
    'drive_manifest':   '/content/drive/MyDrive/DISC5/disc5_recording_manifest.csv',
    'drive_epoch_sets': '/content/drive/MyDrive/DISC5/epoch_sets',     # NEW: 25 per-epoch CSVs + manifest
    'drive_ckpt_dir':   '/content/drive/MyDrive/DISC5_Checkpoints',
    'ssd_root':         '/content/disc5/tensors',
    'output_dir':       '/content/output',
    'tag':              'disc5_arcface_8k',

    'pilot':          False,   # True = ~pilot_hulls train hulls (quick dry run); False = full 646
    'pilot_hulls':    20,

    'sk_kernels':  [127,511,2047,8191],
    'sk_channels': 64,
    'embed_dim':   512,
    'arc_s':       30.0,
    'arc_m':       0.3,

    'seg_samples': 40000,

    'epochs':          40,
    'batch_size':      16,    # physical (peak ~75 GB est.; BS=8 peaked 37.8 GB on 95 GB card)
    'target_eff_batch':64,    # accum = 64 // 16 = 4
    'lr':              1e-3,
    'lr_min':          1e-6,
    'warmup_epochs':   3,
    'warmup_lr_start': 1e-4,
    'weight_decay':    1e-4,
    'grad_clip':       10.0,
    'ckpt_every':      3,
    'patience':        10,

    'num_workers': 8,
    'pin_memory':  True,
    'seed':        1234,
    'resume':      '/content/drive/MyDrive/DISC5_Checkpoints/disc5_arcface_8k_best.pth',

    # --- training data: 25 pre-balanced epoch-set CSVs (originals-preferred policy) ---
    # Generated by disc5_build_epoch_sets.py; per-epoch CSV chosen by (epoch-1) % n_epoch_sets.
    # No keep_aug/cap; no WeightedRandomSampler. Loader rebuilt per epoch in Cell 8.
    'n_epoch_sets':   25,
}
OUT_DIR = Path(CFG['output_dir'])/CFG['tag']; OUT_DIR.mkdir(parents=True,exist_ok=True)
Path(CFG['drive_ckpt_dir']).mkdir(parents=True,exist_ok=True)
with open(OUT_DIR/'config.json','w') as f: json.dump(CFG,f,indent=2)
print(json.dumps(CFG,indent=2))

# Check which best-checkpoint file exists on Drive (for the resume path)
import os
ckpt_dir = CFG['drive_ckpt_dir']
print([f for f in os.listdir(ckpt_dir) if 'best' in f])


In [ ]:
# Cell 3 — Load tensors to local SSD from the Drive tar (live dashboard)
# Two phases with a boxed dashboard matching dash() style: COPY (Drive -> /content)
# then EXTRACT (tar -> SSD). Skips both if the SSD already has all 322,159 tensors.
# Chunked manual copy + post-copy size verification catches the silent Drive-FUSE
# truncation that broke the previous session (shutil.copy2 returned in 1.2 min for
# a "52 GB" copy that wasn't, with no error). Always refreshes the small CSVs
# (tensor_index, manifest, epoch_sets) at the end, and asserts the final counts.
import os, time, tarfile, shutil
from IPython.display import clear_output

SSD = CFG['ssd_root']; os.makedirs(SSD, exist_ok=True)
EXPECTED = 322_159

def count_npy(root):
    return sum(1 for _,_,fs in os.walk(root) for f in fs if f.endswith('.npy'))

n_have = count_npy(SSD)
if n_have >= EXPECTED:
    print(f'SSD already has {n_have:,} tensors -- skip load.')
else:
    # ---- dashboard helpers (match dash() box style in Cell 7) ----
    W=82; T='─'; V='│'; TL='┌'; TR='┐'
    BL='└'; BR='┘'; ML='├'; MR='┤'
    row = lambda s: f'{V}  {s:<{W-4}s}{V}'
    sep = lambda l='': (f'{ML}{T} {l} {T*(W-len(l)-4)}{MR}' if l else f'{ML}{T*(W-2)}{MR}')
    def fmt_time(s):
        h=int(s//3600); m=int((s%3600)//60); sec=int(s%60)
        return (f'{h}h {m:02d}m {sec:02d}s' if h else f'{m}m {sec:02d}s')
    def pbar(frac, w=50):
        fw = int(w*max(0.0, min(1.0, frac)))
        return '█'*fw + '░'*(w-fw)
    def render(lines):
        clear_output(wait=True)
        title = ' DISC5 SSD load '
        L = [f'{TL}{T}{title}{T*(W-len(title)-3)}{TR}'] + lines + [f'{BL}{T*(W-2)}{BR}']
        print('\n'.join(L))

    # ---- Phase 1: COPY Drive -> /content (chunked + size-verified) ----
    src = CFG['drive_tar']
    dst = '/content/disc5_tensors.tar'
    total = os.path.getsize(src)
    CHUNK = 64 * 1024 * 1024
    done = 0; t0 = time.time(); last = 0.0
    with open(src, 'rb') as fi, open(dst, 'wb') as fo:
        while True:
            buf = fi.read(CHUNK)
            if not buf: break
            fo.write(buf); done += len(buf)
            now = time.time()
            if now - last >= 2.0 or done == total:
                el = now - t0; rate = done/el if el>0 else 0
                eta = (total-done)/rate if rate>0 else 0
                frac = done/total
                render([
                    row('Phase 1: COPY    Drive -> /content'),
                    sep(),
                    row(f'[{pbar(frac)}] {frac*100:5.1f}%'),
                    row(f'{done/1e9:6.2f} / {total/1e9:6.2f} GB   '
                        f'{rate/1e6:6.1f} MB/s   ETA {fmt_time(eta)}'),
                ])
                last = now
    copy_time = time.time() - t0
    # Catch silent Drive-FUSE truncation (the previous session's failure mode).
    loc_sz = os.path.getsize(dst)
    if loc_sz != total:
        raise RuntimeError(f'COPY TRUNCATED: local {loc_sz:,} != Drive {total:,} '
                           f'(short by {(total-loc_sz)/1e9:.2f} GB) -- re-run cell.')

    # ---- Phase 2: EXTRACT tar -> SSD ----
    t0 = time.time(); last = 0.0
    render([
        row(f'Phase 1: COPY    done in {fmt_time(copy_time)}   {total/1e9:.1f} GB'),
        sep(),
        row('Phase 2: EXTRACT opening tar...'),
    ])
    with tarfile.open(dst) as tf:
        for i, m in enumerate(tf, 1):
            tf.extract(m, path=SSD, filter='data')
            now = time.time()
            if now - last >= 2.0:
                el = now - t0; rate = i/el if el>0 else 0
                eta = (EXPECTED-i)/rate if rate>0 else 0
                frac = i/EXPECTED
                render([
                    row(f'Phase 1: COPY    done in {fmt_time(copy_time)}   {total/1e9:.1f} GB'),
                    sep(),
                    row('Phase 2: EXTRACT tar -> SSD'),
                    row(f'[{pbar(frac)}] {frac*100:5.1f}%'),
                    row(f'{i:,} / {EXPECTED:,} files   '
                        f'{rate:6.0f} files/s   ETA {fmt_time(eta)}'),
                ])
                last = now

# ---- always: refresh small CSVs + final integrity check ----
shutil.copy2(f"{CFG['drive_tensors']}/tensor_index.csv", f'{SSD}/tensor_index.csv')
shutil.copy2(CFG['drive_manifest'], f'{SSD}/disc5_recording_manifest.csv')

# NEW: refresh epoch_sets directory (25 per-epoch CSVs + manifest). Always copied
# so a rebuild on Drive takes effect on the next notebook start.
es_src = CFG['drive_epoch_sets']
es_dst = f'{SSD}/epoch_sets'; os.makedirs(es_dst, exist_ok=True)
es_files = sorted(f for f in os.listdir(es_src) if f.endswith('.csv'))
for f in es_files:
    shutil.copy2(f'{es_src}/{f}', f'{es_dst}/{f}')
expected_es = CFG['n_epoch_sets'] + 1  # 25 epoch_set_NN.csv + 1 manifest
print(f'epoch_sets/: refreshed {len(es_files)} CSV(s)  (expected {expected_es})')
assert len(es_files) == expected_es, f'epoch_sets count {len(es_files)} != {expected_es}'

n_final = count_npy(SSD)
print(f'SSD tensors: {n_final:,}  (expected {EXPECTED:,})')
assert n_final >= EXPECTED, f'INTEGRITY FAIL: {n_final} < {EXPECTED}'


In [ ]:
# Cell 4 — imports, identity map, passage grouping, datasets, val loader
# Replaces D47 thinning + WeightedRandomSampler with the 25 pre-balanced epoch
# sets (D50). Per-identity balance is achieved by the build script (constant
# per-hull contribution per set); the train loader is rebuilt per epoch in
# Cell 8 from the right epoch_set_NN.csv. val_loader is built here, once.
import csv, time, numpy as np, torch, math
import torch.nn as nn, torch.nn.functional as F, pandas as pd
from pathlib import Path
from collections import Counter, defaultdict
from torch.utils.data import Dataset, DataLoader
SSD=Path(CFG['ssd_root']); device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(CFG['seed']); np.random.seed(CFG['seed'])
with open(SSD/'tensor_index.csv',encoding='utf-8') as f: rows=list(csv.DictReader(f))
man=pd.read_csv(SSD/'disc5_recording_manifest.csv',encoding='utf-8')
def stem_of(r):
    return f"{r['collection']}-{int(r['recording_id']):04d}" if r['source']=='IARA' else Path(str(r['collection'])).stem
stem2session={stem_of(r):str(r['session_id']) for _,r in man.iterrows()}
session_for=lambda pc: stem2session.get(pc,pc)

# train_rows used ONLY to build the label map (full 646 hulls so head width is
# stable across resumes). Per-epoch row lists come from epoch_set CSVs in Cell 8.
train_rows=[r for r in rows if r['split']=='train']
val_rows=[r for r in rows if r['split']=='val' and r['is_augmented']=='0']
if CFG['pilot']:
    th=sorted({r['vessel_id'] for r in train_rows})
    keep=set(np.random.default_rng(CFG['seed']).choice(th,size=min(CFG['pilot_hulls'],len(th)),replace=False))
    train_rows=[r for r in train_rows if r['vessel_id'] in keep]
    print(f'PILOT: {len(keep)} train hulls ({len(train_rows)} tensors); val kept full')

train_ids=sorted({r['vessel_id'] for r in train_rows}); id2lab={v:i for i,v in enumerate(train_ids)}; N_ID=len(train_ids)
print(f'train identities (head width): {N_ID} | full train tensors {len(train_rows)} | val tensors {len(val_rows)}')

class TrainDS(Dataset):
    def __init__(self,rws): self.paths=[SSD/r['path'] for r in rws]; self.labels=[id2lab[r['vessel_id']] for r in rws]
    def __len__(self): return len(self.paths)
    def __getitem__(self,i): return torch.from_numpy(np.load(self.paths[i]).astype('float32').reshape(1,-1)), self.labels[i]
class ValDS(Dataset):
    def __init__(self,rws,preload=True):
        self.vessel=[r['vessel_id'] for r in rws]; self.passage=[session_for(r['parent_clip']) for r in rws]
        self.paths=[SSD/r['path'] for r in rws]; self.preload=preload
        if preload: self.cache=[np.load(p).astype('float32').reshape(1,-1) for p in self.paths]
    def __len__(self): return len(self.paths)
    def __getitem__(self,i):
        x=self.cache[i] if self.preload else np.load(self.paths[i]).astype('float32').reshape(1,-1)
        return torch.from_numpy(x), i

val_ds=ValDS(val_rows,preload=True)
val_loader=DataLoader(val_ds,batch_size=CFG['batch_size'],shuffle=False,
                     num_workers=CFG['num_workers'],pin_memory=CFG['pin_memory'])
print(f'val hulls {len(set(val_ds.vessel))} | val passages {len(set(val_ds.passage))}')

# Read epoch_sets plan fingerprint from the manifest header (audit trail).
with open(SSD/'epoch_sets/epoch_sets_manifest.csv', encoding='utf-8') as f:
    epoch_sets_fp_header = f.readline().strip().lstrip('#').strip()
print(f'epoch_sets plan: {epoch_sets_fp_header}')

# Preview the first epoch_set so Cell 5b can report expected batches/epoch.
with open(SSD/'epoch_sets/epoch_set_01.csv', encoding='utf-8') as f:
    _ep1 = list(csv.DictReader(f))
ROWS_PER_SET_PREVIEW = len(_ep1)
print(f'epoch_set_01.csv preview: {ROWS_PER_SET_PREVIEW:,} rows')


In [ ]:
# Cell 4b — record val tensors + epoch_sets fingerprint for posterity/repro
# Replaces the old thinning-fingerprint dump (D47). The 25 epoch_set CSVs are
# themselves the authoritative training-data record (already on Drive); this
# cell just persists the val list and the plan fingerprint alongside checkpoints.
import csv, json, hashlib
from pathlib import Path

REC = Path(CFG['drive_ckpt_dir']); REC.mkdir(parents=True, exist_ok=True)
tag = CFG['tag']

def _dump(rows, path):
    cols = ['path','vessel_id','source','split','parent_clip','seg_idx','is_augmented']
    with open(path,'w',newline='',encoding='utf-8') as f:
        w=csv.writer(f); w.writerow(cols)
        for r in rows: w.writerow([r[c] for c in cols])

val_manifest = REC/f'{tag}_val_tensors_used.csv'
_dump(val_rows, val_manifest)

meta = {
    'tag': tag, 'seed': CFG['seed'],
    'n_epoch_sets': CFG['n_epoch_sets'],
    'epoch_sets_fingerprint_header': epoch_sets_fp_header,
    'n_train_hulls': N_ID,
    'n_val_tensors': len(val_rows),
    'n_val_hulls': len({r['vessel_id'] for r in val_rows}),
    'rows_per_set_preview': ROWS_PER_SET_PREVIEW,
}
with open(REC/f'{tag}_tensors_used_meta.json','w') as f: json.dump(meta,f,indent=2)

print(json.dumps(meta, indent=2))
print('wrote:', val_manifest.name, '| meta json')


In [ ]:
# Cell 5 — Model + ArcFace head
class SKFilterbank(nn.Module):
    def __init__(self,in_ch=1,out_ch=64,kernels=(127,511,2047,8191),d=4):
        super().__init__(); self.convs=nn.ModuleList(); self.norms=nn.ModuleList(); ng=min(16,out_ch)
        for k in kernels: self.convs.append(nn.Conv1d(in_ch,out_ch,k,padding=k//2)); self.norms.append(nn.GroupNorm(ng,out_ch))
        self.squeeze=nn.Linear(out_ch,d); self.excites=nn.ModuleList([nn.Linear(d,out_ch) for _ in kernels])
    def forward(self,x):
        outs=[F.relu(n(c(x))) for c,n in zip(self.convs,self.norms)]
        L=min(o.shape[-1] for o in outs); outs=[o[...,:L] for o in outs]
        st=torch.stack(outs,0); U=st.sum(0); z=F.relu(self.squeeze(U.mean(-1)))
        a=F.softmax(torch.stack([e(z) for e in self.excites],0),0).unsqueeze(-1)
        return (st*a).sum(0)
class DISC5Encoder(nn.Module):
    def __init__(self,kernels,sk_ch=64,embed_dim=512):
        super().__init__(); self.fb=SKFilterbank(1,sk_ch,tuple(kernels))
        self.l1=nn.Sequential(nn.Conv2d(1,64,3,stride=(1,1),padding=1),nn.GroupNorm(16,64),nn.ReLU(True))
        self.l2=nn.Sequential(nn.Conv2d(64,128,3,stride=(1,4),padding=1),nn.GroupNorm(16,128),nn.ReLU(True))
        self.l3=nn.Sequential(nn.Conv2d(128,256,3,stride=(1,4),padding=1),nn.GroupNorm(16,256),nn.ReLU(True))
        self.l4=nn.Sequential(nn.Conv2d(256,512,3,stride=(2,2),padding=1),nn.GroupNorm(16,512),nn.ReLU(True))
        self.l5=nn.Sequential(nn.Conv2d(512,embed_dim,3,stride=(2,2),padding=1),nn.GroupNorm(16,embed_dim),nn.ReLU(True))
        self.pool=nn.AdaptiveAvgPool2d(1)
    def forward(self,x):
        h=self.fb(x).unsqueeze(1)
        for l in (self.l1,self.l2,self.l3,self.l4,self.l5): h=l(h)
        return F.normalize(self.pool(h).flatten(1),dim=1)
class ArcFace(nn.Module):
    def __init__(self,in_dim,n_cls,s=30.0,m=0.3):
        super().__init__(); self.W=nn.Parameter(torch.empty(n_cls,in_dim)); nn.init.xavier_normal_(self.W); self.s,self.m=s,m
    def forward(self,emb,labels):
        cos=(emb@F.normalize(self.W,dim=1).t()).clamp(-1+1e-7,1-1e-7)
        return self.s*torch.cos(torch.acos(cos)+self.m*F.one_hot(labels,self.W.size(0)).float())
if device.type=='cuda': torch.backends.cudnn.benchmark=True; print('GPU:',torch.cuda.get_device_name(0))
model=DISC5Encoder(CFG['sk_kernels'],CFG['sk_channels'],CFG['embed_dim']).to(device).float()
head=ArcFace(CFG['embed_dim'],N_ID,CFG['arc_s'],CFG['arc_m']).to(device)
print(f'params: {sum(p.numel() for p in model.parameters())+sum(p.numel() for p in head.parameters()):,} | head {N_ID}x{CFG["embed_dim"]} (discarded at inference)')


In [ ]:
# Cell 5b — physical batch + gradient-accumulation setup (no loader build)
# Loader is built per epoch in Cell 8 from the epoch_set CSV. Expected batches
# below are estimated from epoch_set_01.csv; actual count per epoch may differ
# slightly across sets (constant per build, ~63,300 rows / BS).
PHYS_BS = CFG['batch_size']                            # 16
ACCUM   = max(1, CFG['target_eff_batch'] // PHYS_BS)   # 64 // 16 = 4
expected_batches = ROWS_PER_SET_PREVIEW // PHYS_BS
expected_opt_steps = expected_batches // ACCUM
print(f'physical batch={PHYS_BS} | accum={ACCUM} | effective={PHYS_BS*ACCUM}')
print(f'expected train batches/epoch~{expected_batches:,} '
      f'(optimizer steps/epoch ~{expected_opt_steps:,})')


In [ ]:
# Cell 6 — Optimizer + scheduler + criterion + resume
params=list(model.parameters())+list(head.parameters())
optimizer=torch.optim.AdamW(params,lr=CFG['warmup_lr_start'],weight_decay=CFG['weight_decay'])
def lr_lambda(ep):
    wu=CFG['warmup_epochs']
    if ep<wu: return 1.0+(CFG['lr']/CFG['warmup_lr_start']-1.0)*(ep/wu)
    prog=(ep-wu)/max(1,CFG['epochs']-wu)
    return (CFG['lr_min']+0.5*(CFG['lr']-CFG['lr_min'])*(1+math.cos(math.pi*prog)))/CFG['warmup_lr_start']
scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda)
criterion=nn.CrossEntropyLoss()
start_epoch=1; history=[]; best_gap=-1.0; best_epoch=0; patience=0
if CFG['resume']:
    ck=torch.load(CFG['resume'],map_location=device,weights_only=False)
    model.load_state_dict(ck['model']); head.load_state_dict(ck['head'])
    optimizer.load_state_dict(ck['optimizer']); scheduler.load_state_dict(ck['scheduler'])
    start_epoch=ck['epoch']+1; best_gap=ck.get('best_gap',-1.0); history=ck.get('history',[])
    # best_epoch is a new field. Legacy best.pth lacks it, but best.pth is written AT the
    # best epoch, so its own 'epoch' is the correct fallback for this resume.
    best_epoch=ck.get('best_epoch',ck['epoch'])
    print(f'Resumed at epoch {start_epoch}, best_gap={best_gap:.4f} (best epoch {best_epoch})')
print('Optimizer/scheduler/criterion ready.')


In [ ]:
# Cell 7 — metrics, dashboard, epoch fns (gradient accumulation; re-ID metrics)
from sklearn.metrics import roc_curve
def fmt_time(s):
    h=int(s//3600); m=int((s%3600)//60); sec=int(s%60); return (f'{h}h {m:02d}m {sec:02d}s' if h else f'{m}m {sec:02d}s')
def get_vram():
    if not torch.cuda.is_available(): return 0.0,0.0
    return torch.cuda.max_memory_allocated()/1e9, torch.cuda.get_device_properties(0).total_memory/1e9
@torch.no_grad()
def embed_val(model,loader):
    model.eval(); E=[]
    for x,_ in loader: E.append(model(x.to(device)).cpu())
    return torch.cat(E,0)
def passage_pool(seg_emb,vessels,passages):
    by=defaultdict(list)
    for i,p in enumerate(passages): by[p].append(i)
    pid=list(by); pv=torch.stack([F.normalize(seg_emb[by[p]].mean(0),dim=0) for p in pid])
    pves=np.array([vessels[by[p][0]] for p in pid]); return pid,pv,pves
def compute_reid(seg_emb,vessels,passages,seg_pairs=20000,rng=None):
    rng=rng or np.random.default_rng(0)
    pid,pv,pves=passage_pool(seg_emb,vessels,passages); S=(pv@pv.t()).numpy(); P=len(pid)
    iu=np.triu_indices(P,1); same=(pves[:,None]==pves[None,:]); ps=S[iu]; py=same[iu].astype(int)
    sc=ps[py==1].mean() if (py==1).any() else float('nan'); dc=ps[py==0].mean() if (py==0).any() else float('nan')
    diff_lt05=float((ps[py==0]<0.5).mean()) if (py==0).any() else float('nan')
    if (py==1).any() and (py==0).any():
        fpr,tpr,_=roc_curve(py,ps); fnr=1-tpr; eer=float(fpr[np.nanargmin(np.abs(fnr-fpr))])
    else: eer=float('nan')
    np.fill_diagonal(S,-2.0); nn_idx=S.argmax(1); vc=Counter(pves.tolist())
    elig=np.array([vc[v]>=2 for v in pves]); r1=float((pves[nn_idx][elig]==pves[elig]).mean()) if elig.any() else float('nan')
    n=len(vessels); ves=np.array(vessels); a=rng.integers(0,n,seg_pairs); b=rng.integers(0,n,seg_pairs); ok=a!=b; a,b=a[ok],b[ok]
    scr=(seg_emb[a]*seg_emb[b]).sum(1).numpy(); sy=(ves[a]==ves[b]).astype(int)
    return dict(same=sc,diff=dc,gap=sc-dc,eer=eer,rank1=r1,diff_lt05=diff_lt05,
                seg_same=scr[sy==1].mean() if (sy==1).any() else float('nan'),
                seg_diff=scr[sy==0].mean() if (sy==0).any() else float('nan'),n_pass=P)
def baseline_gap_stub():
    """D38 explicit-feature (tonal/shaft/blade) NN baseline. Deferred -> None."""
    return None
def train_one_epoch(model,head,loader,opt,crit,ep,clip,accum,log_every=100):
    model.train(); head.train(); tot=0;n=0;gns=[]; t0=time.time(); nb=len(loader); opt.zero_grad(set_to_none=True)
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    s_sum=s_cnt=d_sum=d_cnt=0.0
    for step,(x,y) in enumerate(loader):
        x,y=x.to(device),y.to(device); emb=model(x); raw=crit(head(emb,y),y)
        (raw/accum).backward(); tot+=raw.item()*x.size(0); n+=x.size(0)
        if (step+1)%accum==0 or (step+1)==nb:
            gn=torch.nn.utils.clip_grad_norm_(list(model.parameters())+list(head.parameters()),clip)
            gns.append(float(gn)); opt.step(); opt.zero_grad(set_to_none=True)
        with torch.no_grad():
            Cm=emb@emb.t(); same=(y[:,None]==y[None,:]); eye=torch.eye(len(y),dtype=torch.bool,device=y.device)
            sm=same&~eye; dm=~same
            if sm.any(): s_sum+=Cm[sm].sum().item(); s_cnt+=int(sm.sum())
            if dm.any(): d_sum+=Cm[dm].sum().item(); d_cnt+=int(dm.sum())
        if (step+1)%log_every==0 or step+1==nb:
            el=time.time()-t0; ips=(step+1)/el if el>0 else 0
            eta=(nb-(step+1))/ips if ips>0 else 0
            eh=int(eta//3600); em=int((eta%3600)//60); es=int(eta%60)
            etastr=(f'{eh}h{em:02d}m' if eh else (f'{em}m{es:02d}s' if em else f'{es}s'))
            g=gns[-1] if gns else float('nan')
            if torch.cuda.is_available():
                pk=torch.cuda.max_memory_allocated()/1e9; tot_v=torch.cuda.get_device_properties(0).total_memory/1e9
            else:
                pk=tot_v=0.0
            print(f'  ep{ep:02d} [{step+1:>5d}/{nb}] loss={raw.item():.4f} grad={g:.2f} '
                  f'{ips:.1f} it/s ETA {etastr} VRAM {pk:.1f}/{tot_v:.0f} GB')
    ts=s_sum/s_cnt if s_cnt else float('nan'); td=d_sum/d_cnt if d_cnt else float('nan')
    return dict(loss=tot/n, grad=float(np.mean(gns)) if gns else 0.0, tr_same=ts, tr_diff=td, tr_gap=ts-td)
def dash(ep,cfg,tr,m,lr,best,pat,is_best,ep_t,tot,best_ep=None):
    W=82;T='─';V='│';TL='┌';TR='┐';BL='└';BR='┘';ML='├';MR='┤'
    row=lambda s:f'{V}  {s:<{W-4}s}{V}'; sep=lambda l='':(f'{ML}{T} {l} {T*(W-len(l)-4)}{MR}' if l else f'{ML}{T*(W-2)}{MR}')
    pk,ttl=get_vram(); pct=ep/cfg['epochs']; fw=int(50*pct); bar='█'*fw+'░'*(50-fw)
    eta=tot/ep*(cfg['epochs']-ep) if ep>0 else 0
    spread=tr['tr_gap']-m['gap']
    L=[f'{TL}{T} DISC5 ArcFace 8kHz re-ID {T*(W-27)}{TR}']
    L+=[row(f'GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}  '
            f'VRAM peak {pk:.1f}/{ttl:.0f} GB  physBS {cfg["batch_size"]} x accum {ACCUM} = eff {cfg["batch_size"]*ACCUM}')]
    L+=[sep('PROGRESS'),row(f'[{bar}] {pct*100:5.1f}%'),
        row(f'Epoch {ep:>3d}/{cfg["epochs"]}  Ep {fmt_time(ep_t)}  Elapsed {fmt_time(tot)}  ETA {fmt_time(eta)}')]
    L+=[sep('VAL RE-ID (passage-level = primary)'),
        row(f'same {m["same"]:.4f}   diff {m["diff"]:.4f}   GAP {m["gap"]:+.4f}{" *" if is_best else ""}'),
        row(f'EER {m["eer"]:.4f}   rank-1 {m["rank1"]:.4f}   passages {m["n_pass"]}   diff<0.5 {m["diff_lt05"]:.2f}'),
        row(f'seg-level (diag): same {m["seg_same"]:.4f}  diff {m["seg_diff"]:.4f}')]
    L+=[sep('TRAIN'),row(f'loss {tr["loss"]:.4f}   grad {tr["grad"]:.2f}   LR {lr:.2e}'),
        row(f'train GAP {tr["tr_gap"]:+.4f}  (train-val spread {spread:+.4f}  larger = more overfit)'),
        row(f'best val GAP {best:+.4f}{f" (ep{best_ep})" if best_ep else ""}   patience {pat}/{cfg["patience"]}')]
    L+=[sep('HISTORY (last 10)'),row(f'{"ep":>3s} {"loss":>7s} {"same":>7s} {"diff":>7s} {"gap":>7s} {"EER":>6s} {"r1":>6s}')]
    for h in history[-10:]:
        L+=[row(f'{h["epoch"]:>3d} {h["loss"]:>7.4f} {h["same"]:>7.4f} {h["diff"]:>7.4f} {h["gap"]:>+7.4f} {h["eer"]:>6.3f} {h["rank1"]:>6.3f}')]
    L+=[f'{BL}{T*(W-2)}{BR}']; print('\n'.join(L))
print('metrics + dashboard + epoch fns ready.')


In [ ]:
# Cell 7b — Optional mid-run VRAM clear (run before training if a prior cell OOMed,
# to avoid re-copying tensors to SSD on a full runtime restart)
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    fr,to=torch.cuda.mem_get_info(); print(f'VRAM after clear: {fr/1e9:.1f}/{to/1e9:.1f} GB free')


In [ ]:
# Cell 8 — Training loop (accum; ckpt every 3 + best by passage gap; SSD->Drive; early stop)
# Periodic resumable checkpoint every CFG['ckpt_every'] epochs -> Drive (latest is
# all you need to resume; <=3 epochs lost on a crash). Best-by-gap writes best.pth
# + best_val_seg_emb.npy to Drive (the .npy is for the diagnostics cells). The full
# per-epoch metric history is embedded in every checkpoint, so epoch curves never
# need every epoch's weights. Live dashboard each epoch via dash().
# D50: train loader is rebuilt per epoch from epoch_set_NN.csv (no sampler).
import shutil
t_start=time.time()
DRIVE_CK=Path(CFG['drive_ckpt_dir']); DRIVE_CK.mkdir(parents=True,exist_ok=True)
def save_ckpt(path,ep,m):
    torch.save(dict(epoch=ep,model=model.state_dict(),head=head.state_dict(),
                    optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),
                    best_gap=best_gap,best_epoch=best_epoch,history=history,config=CFG,val=m),path)
for epoch in range(start_epoch,CFG['epochs']+1):
    # ---- D50: build this epoch's train loader from epoch_set_NN.csv ----
    set_idx = ((epoch - 1) % CFG['n_epoch_sets']) + 1
    epoch_csv = SSD/f'epoch_sets/epoch_set_{set_idx:02d}.csv'
    with open(epoch_csv, encoding='utf-8') as f:
        epoch_rows = list(csv.DictReader(f))
    train_ds = TrainDS(epoch_rows)
    train_loader = DataLoader(train_ds, batch_size=PHYS_BS, shuffle=True,
                              num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'],
                              drop_last=True, persistent_workers=False)
    if epoch == start_epoch:
        print(f'epoch {epoch}: loaded epoch_set_{set_idx:02d}.csv  '
              f'({len(epoch_rows):,} rows, {len(train_loader):,} batches)')

    ep_t0=time.time()
    tr=train_one_epoch(model,head,train_loader,optimizer,criterion,epoch,CFG['grad_clip'],ACCUM)
    seg_emb=embed_val(model,val_loader)
    m=compute_reid(seg_emb,val_ds.vessel,val_ds.passage,rng=np.random.default_rng(CFG['seed']))
    current_lr=optimizer.param_groups[0]['lr']; scheduler.step()
    ep_t=time.time()-ep_t0; tot=time.time()-t_start
    is_best=m['gap']>best_gap
    if is_best: best_gap=m['gap']; best_epoch=epoch; patience=0
    else: patience+=1
    history.append(dict(epoch=epoch,loss=tr['loss'],**{k:m[k] for k in ('same','diff','gap','eer','rank1')}))
    dash(epoch,CFG,tr,m,current_lr,best_gap,patience,is_best,ep_t,tot,best_epoch)
    # --- best: checkpoint + best-epoch val embeddings, both to Drive ---
    if is_best:
        save_ckpt(OUT_DIR/'best.pth',epoch,m)
        np.save(OUT_DIR/'best_val_seg_emb.npy',seg_emb.numpy())
        shutil.copy2(OUT_DIR/'best.pth',DRIVE_CK/f'{CFG["tag"]}_best.pth')
        np.save(DRIVE_CK/f'{CFG["tag"]}_best_val_seg_emb.npy',seg_emb.numpy())
    # --- periodic resumable checkpoint every ckpt_every epochs -> Drive ---
    if epoch%CFG['ckpt_every']==0:
        local_ep=OUT_DIR/f'epoch_{epoch:03d}.pth'; save_ckpt(local_ep,epoch,m)
        shutil.copy2(local_ep,DRIVE_CK/f'{CFG["tag"]}_ep{epoch:03d}.pth')
    if patience>=CFG['patience']:
        print(f'\nEarly stop at epoch {epoch} (no gap improvement {CFG["patience"]} epochs)'); break
print(f'\nDone. Best passage-level same/diff gap: {best_gap:+.4f}')
b=baseline_gap_stub(); print('D38 baseline gap:', 'DEFERRED (stub)' if b is None else f'{b:+.4f}')


In [ ]:
# Cell 9 — Short post-training diagnostics (passage-level cosine + epoch curves)
# Uses the BEST checkpoint's val embeddings (best_val_seg_emb.npy from Cell 8) and
# the in-memory val_ds + history. Run after Cell 8 in the same session (needs
# Cells 4 & 7 for val_ds + passage_pool). Two plots only: the D24 same-vs-different
# passage-cosine separation (the go/no-go visual) and the val metric trend.
import numpy as np, matplotlib.pyplot as plt

seg = np.load(OUT_DIR/'best_val_seg_emb.npy')          # (n_val_seg, 512), L2-normed
pid, pv, pves = passage_pool(torch.from_numpy(seg), val_ds.vessel, val_ds.passage)
pv = pv.numpy(); P = len(pid)
S = pv @ pv.T; iu = np.triu_indices(P, 1)
same = (pves[:,None]==pves[None,:])[iu]; cos = S[iu]
sc, dc = cos[same], cos[~same]

fig, ax = plt.subplots(1, 2, figsize=(13,4))
bins = np.linspace(-0.2, 1.0, 50)
ax[0].hist(dc, bins=bins, alpha=0.6, density=True, color='tab:red',   label=f'different ({len(dc)})')
ax[0].hist(sc, bins=bins, alpha=0.6, density=True, color='tab:green', label=f'same ({len(sc)})')
ax[0].axvline(0.5, ls='--', c='gray', lw=1)
ax[0].set_title(f'Passage cosine  (gap {sc.mean()-dc.mean():+.3f})')
ax[0].set_xlabel('cosine similarity'); ax[0].set_ylabel('density'); ax[0].legend()

ep = [h['epoch'] for h in history]
ax[1].plot(ep, [h['gap']   for h in history], marker='o', label='gap')
ax[1].plot(ep, [h['eer']   for h in history], marker='s', label='EER')
ax[1].plot(ep, [h['rank1'] for h in history], marker='^', label='rank-1')
ax[1].set_title('Validation over epochs'); ax[1].set_xlabel('epoch')
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'same cos: mean {sc.mean():.3f}  std {sc.std():.3f}')
print(f'diff cos: mean {dc.mean():.3f}  std {dc.std():.3f}')
print(f'gap     : {sc.mean()-dc.mean():+.3f}   diff<0.5 frac: {(dc<0.5).mean():.2f}')


In [ ]:
# Cell 10 — Save all artifacts to Drive (for the detailed diagnostics notebook)
# Retired and replaced with cell # 16

In [ ]:
# Cell 11 — within-platform GAP diagnostic (one-shot, against best.pth)
# Splits the val GAP into per-platform subsets to isolate the channel-asymmetry
# effect we measured earlier (95% same-platform same-vessel pairs vs 70%
# same-platform diff-vessel pairs). If the within-OS gap is meaningfully wider
# than the overall gap, the val metric is partly tracking platform mismatch
# rather than identity. If it's similar, the embedding is genuinely failing
# to separate identities. Read-only; does NOT modify training state.

import torch, numpy as np
from collections import Counter

# Per-passage platform from the manifest (session_id -> platform, first seen).
pass_platform = {}
for _, r in man.iterrows():
    sid = str(r['session_id'])
    pass_platform.setdefault(sid, r['platform'])

# Load best.pth weights into a CLONE model so the live model is undisturbed.
diag_model = DISC5Encoder(CFG['sk_kernels'], CFG['sk_channels'], CFG['embed_dim']).to(device).float()
_ck = torch.load(CFG['resume'], map_location=device, weights_only=False)
diag_model.load_state_dict(_ck['model']); diag_model.eval()
print(f"loaded best.pth from epoch {_ck['epoch']} "
      f"(best_gap={_ck.get('best_gap', float('nan')):.4f})")

# Embed val with the diagnostic model.
with torch.no_grad():
    _emb = [diag_model(x.to(device)).cpu() for x,_ in val_loader]
seg_emb = torch.cat(_emb, 0)

# Passage pool + platform vector per passage.
pid, pv, pves = passage_pool(seg_emb, val_ds.vessel, val_ds.passage)
pv = pv.numpy(); P = len(pid)
pplat = np.array([pass_platform.get(str(sid), 'UNKNOWN') for sid in pid])
print('passages by platform:', dict(Counter(pplat)))

# Cosine matrix + upper-triangle labels.
S = pv @ pv.T
iu = np.triu_indices(P, 1)
same      = (pves[:, None] == pves[None, :])[iu]
same_plat = (pplat[:, None] == pplat[None, :])[iu]
cos = S[iu]

def report(label, mask):
    sm = same & mask
    dm = (~same) & mask
    if not sm.any() or not dm.any():
        print(f'  {label:<30s}   insufficient pairs (same={int(sm.sum())}, diff={int(dm.sum())})')
        return
    sc, dc = cos[sm].mean(), cos[dm].mean()
    d05 = (cos[dm] < 0.5).mean()
    print(f'  {label:<30s}   same {sc:.4f} ({int(sm.sum()):4d})   '
          f'diff {dc:.4f} ({int(dm.sum()):5d})   GAP {sc-dc:+.4f}   diff<0.5 {d05:.2f}')

print()
print('Per-slice GAP (pairs restricted to a subset of passages):')
all_mask = np.ones_like(same, dtype=bool)
os_mask  = (pplat == 'OS')
gl_mask  = (pplat == 'Glider')
sh_mask  = (pplat == 'ShipsEar_array')
report('overall (no restriction)',           all_mask)
report('within OS only',                     ((os_mask[:, None]) & (os_mask[None, :]))[iu])
report('within Glider only',                 ((gl_mask[:, None]) & (gl_mask[None, :]))[iu])
report('within ShipsEar_array only',         ((sh_mask[:, None]) & (sh_mask[None, :]))[iu])
report('cross-platform pairs only',          ~same_plat)